<h1>2 - Principaux tests d'hypothèses</h1>

<h2>Setup</h2>

In [5]:
# Standard libraries 
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns


# Statistics libraries
import scipy
from scipy.stats import binomtest

# Scheck a few version
print(f"Pandas version: {pd.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")
print(f"Scipy version: {scipy.__version__}")


Pandas version: 3.0.3
Matplotlib version: 3.11.0
Seaborn version: 0.13.2
Scipy version: 1.18.0


<h2>Test Binomial</h2>

In [7]:
# Load data within a dataframe
df = sns.load_dataset("tips")

# Display dataset
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


<h3><u>Binomtest</u></h3>

Hypothèse HO : "80% des pourboires sont donnés le soir"

In [8]:
# Define the HO probability
h0_p = 0.8

Calculons l'effectif et la porportion de tips (en action de donner, et non en valeur donnée) effectivement réalisées le midi et le soir. Pour cela, on va réaliser une aggregation en comptant le nombre de lignes pour chaque prestation `Dinner` ou `Lunch`. On va donc aggréger la colonne `Time`. La proportion s'obtiens grace au paramètre `normalize` définie à `True`

In [13]:
# Compute the proportion of tips given during the day and the night
print(f'Effectif :\n{df["time"].value_counts()}')
print(f'\nProportion :\n{df["time"].value_counts(normalize=True)}')

Effectif :
time
Dinner    176
Lunch      68
Name: count, dtype: int64

Proportion :
time
Dinner    0.721311
Lunch     0.278689
Name: proportion, dtype: float64


On observe que les clients donnent des pourboire le soir dans 72% des cas. Ce qui est un peu inférieur à l'hypothèse de base. Pour vérifier notre hypothèse, on va avoir besoin de calculer la `P_Value` en réalisant un test Binomial à partir de la méthode `binomtest` de `Scipy`:

``` Python
# Syntaxe
binomtest(k, n,  p=0.5, alternative="two-sided")
```

 <u>Avec</u>:
- `k` (int) : le nombre de succès
- `n` (int) : le nombre d'essais
- `p` (float | optional) : La probabilité hypothétique de succès. Cette valeur doit être comprise entre 0 et 1. Elle est par défaut fixée à 0.5. 
- `alternative` ('two-sided', 'greater', 'less' | optional) : Indique les hypothèses alternatives. La valeur par défaut est "two-sided".  

Lien vers la documentation de la méthode :<a href="https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binomtest.html"> binomtest</a>

In [14]:
# Extract the number of succes
k = df["time"].value_counts()["Dinner"]

# Extract the number of trials
n = len(df)

In [15]:
# Use binomtest module
binomtest(k=k, n=n, p=h0_p)

BinomTestResult(k=176.0, n=244.0, alternative='two-sided', statistic=0.7213114754098361, pvalue=0.002988997747005764)

Le module nous retourne ainsi une `P-Value` de 0,00298.... Ce qui est très faible pour une probabilité. Dans la pratique, on pourra rejeter, ou non, notre hypothèse H0. Cela va dépendre du seuil de risque $\alpha$ qu'on se sera fixé. 

À noter qu'il est courant de prendre $\alpha\ =\ 2\%$

<h3><u>En résumé</u> :</h3>

Voici le protocol à suivre :
- On site notre hypothèse H0
- On définit le seuil de risque $\alpha$ à 2%
- On calcul la P_Value grâce à notre test Binomiale
- On compare notre P_Value à notre seuil de risque $\alpha$ pour conclure

In [18]:
H0_p = 0.8
print(f"H0: \"{H0_p*100} % des pourboires sont donnés le soir\"")
print()

alpha = 0.02
p_value = binomtest(k=k, n=n, p=H0_p).pvalue

if p_value < alpha:
    print("Nous avons suffisement de preuves pour rejeter H0")
else:
    print("Nous n'avons pas suffisement de preuves pour rejeter H0")

H0: "80.0 % des pourboires sont donnés le soir"

Nous avons suffisement de preuves pour rejeter H0


Pour conclure, ici, l'hypothèse H0 aura été réfutée. 

À l'avenir, à chaque fois qu'on se trouve dans une situation ou quelqu'un met sur la table une quelconque théorie à propos d'un pourcentage, donc d'une <span style="color:rgb(249, 116, 50)">variable discrète</span>, alors on peut tester cette théorie à partir d'un <span style="color:rgb(249, 116, 50)">test Binomiale</span>. 